# 00 — Data exploration

Goal: understand the label distributions and input properties **before** choosing prompts and training settings.
All figures are saved to `results/figures/` for the report. `protect-frsemcor` splits are already removed by `load_data()`.

In [ ]:
# Setup: on Colab or Kaggle, clone (or update) the repo so that `src/` is importable; locally, run from the repo root.
import os, sys
REPO_URL = "https://github.com/bkadyrbay-hash/lab1-supersense-llm.git"
BRANCH = "main"  # set to your own branch to test code that is not merged yet
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if "google.colab" in sys.modules or ON_KAGGLE:
    REPO_DIR = ("/kaggle/working" if ON_KAGGLE else "/content") + "/lab1-supersense-llm"
    if not os.path.exists(REPO_DIR):
        !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import re
import matplotlib.pyplot as plt
import pandas as pd
from src.data import load_data, SUPERSENSES, HYPERSENSES

SPLITS = ["train", "rand-dev", "rand-test", "freq-dev", "freq-test"]
FIG_DIR = "results/figures"
df = load_data()

## 1. Split sizes

In [ ]:
df["set"].value_counts().reindex(SPLITS).to_frame("n_senses")

## 2. Label distributions per split

Splits have very different sizes, so we compare **proportions** (each column sums to 1) rather than counts.

In [ ]:
def plot_label_heatmap(labels, col, path):
    """Heatmap of label proportions (rows) per split (columns), saved to `path`."""
    props = pd.crosstab(df[col], df["set"], normalize="columns").reindex(index=labels, columns=SPLITS)
    fig, ax = plt.subplots(figsize=(7, 0.32 * len(labels) + 1.2))
    im = ax.imshow(props, cmap="Blues", aspect="auto", vmin=0)
    ax.set_xticks(range(len(SPLITS)), SPLITS, rotation=30, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    for i, j in [(i, j) for i in range(len(labels)) for j in range(len(SPLITS))]:
        v = props.iat[i, j]
        ax.text(j, i, f"{v:.0%}" if v >= 0.005 else ("<1%" if v > 0 else ""), ha="center", va="center", fontsize=7,
                color="white" if v > 0.6 * props.values.max() else "black")
    fig.colorbar(im, ax=ax, format=lambda x, _: f"{x:.0%}", fraction=0.04)
    ax.set_title(f"{col.capitalize()} proportion per split")
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()
    return props

super_props = plot_label_heatmap(SUPERSENSES, "supersense", f"{FIG_DIR}/supersense_by_split.png")

In [ ]:
hyper_props = plot_label_heatmap(HYPERSENSES, "hypersense", f"{FIG_DIR}/hypersense_by_split.png")

## 3. Properties of the inputs

In [ ]:
# Shallow cues that a classifier (or an LLM) may rely on, besides the meaning of the definition.
df["n_words"] = df["definition"].str.split().str.len()
df["lemma_in_def"] = [l.lower() in d.lower() for l, d in zip(df["lemma"], df["definition"])]
df["starts_synonyme_variante"] = df["definition"].str.match(r"(Synonyme|Variante) de", flags=re.I)
df["in_frsemcor"] = df["nb_occ_lemma_frsemcor"] > 0

input_stats = df.groupby("set")[["n_words", "lemma_in_def", "starts_synonyme_variante", "in_frsemcor"]].mean().reindex(SPLITS)
input_stats["n_empty_def"] = df[df["definition"] == ""].groupby("set").size().reindex(SPLITS, fill_value=0)
input_stats.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.boxplot([df.loc[df["set"] == s, "n_words"] for s in SPLITS], showfliers=False)
ax.set_xticks(range(1, len(SPLITS) + 1), SPLITS)
ax.set_ylabel("definition length (words)")
ax.set_title("Definition length per split (outliers hidden)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/definition_length.png", dpi=200)
plt.show()

## 4. Shortest definitions (likely hard cases)

In [ ]:
df[df["definition"] != ""].nsmallest(10, "n_words")[["lemma", "definition", "supersense", "set"]]

## 5. Take-aways for the experiments

- **The splits come from two different samplings.** `rand-*` are random Wiktionary senses: 26–33 % are `person` (≈18 % of all `rand-*` senses are demonyms, *Habitant de …*). `freq-*` are senses of frequent lemmas (84–88 % appear in FrSemCor vs 14–27 % in `rand-*`) and are dominated by `act`, `artifact`, `cognition` and `other`. `train` is in between.
- Consequence: **macro-F1 is our main metric**, and results on `freq-test` vs `rand-test` must be discussed separately — a configuration tuned on `rand-dev` may not transfer to `freq-test`.
- `rand-dev` contains **no `relation` instance** and only 1–4 instances of several classes: per-class scores on `rand-dev` are noisy, so we compare configurations on macro-F1 *and* accuracy.
- Definitions are short (median 10 words); 19 senses have an empty definition — for these, only the lemma can help, which is one argument for the *with lemma* setting.
- The lemma is rarely copied in its own definition (4–12 %), so providing it adds real information (it is not redundant).